# COMP5122M Coursework 1 — P1 Data Quality Notebook

**Task 31: fresh-kernel, top-to-bottom execution.** Run this notebook after Task 29 exports `data/clean/pew_clean_p1_provisional.csv`. All checks below use the **provisional P1 export** and the original companion CSV. No P2 `educ3` file is needed or silently imputed.

The notebook prints only **aggregate results**, not respondent-level records. Do not commit survey microdata, the executed notebook, or the private audit outputs. This is a verification notebook: the validated export is *read-only* here; the notebook does not overwrite it.

## A. Start clean and verify source integrity

Find the repository from either the root or `notebooks/`; check the Task 29 export against its SHA-256 manifest. A stale or altered export causes an error.

In [ ]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "validate_task29.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "validate_task29.py").is_file(), "Launch notebook from repository root or notebooks/"
sys.path.insert(0, str(ROOT))

CLEAN = ROOT / "data" / "clean"
EXPORT = CLEAN / "pew_clean_p1_provisional.csv"
MANIFEST = CLEAN / "task29_provisional_export_manifest.csv"
SOURCE_CSV = ROOT / "data" / "raw" / "Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv"
REFERENCE = ROOT / "references" / "topline_expected.csv"
for file in (EXPORT, MANIFEST, SOURCE_CSV, REFERENCE):
    assert file.is_file(), f"Missing required input: {file}. Run Task 29 or restore this reference."

manifest = pd.read_csv(MANIFEST)
assert len(manifest) == 1 and "sha256" in manifest.columns, "Invalid Task 29 manifest"
digest = hashlib.sha256(EXPORT.read_bytes()).hexdigest()
assert digest == str(manifest.loc[0, "sha256"]), "P1 export differs from Task 29's recorded checksum"
print("PASS: Task 29 provisional export SHA-256 verified")
print("P2 education mapping: pending; no educ3 values will be invented")

## B. Recheck dimensions and harmonised variables

This section verifies the counts established in Tasks 10–18 and 29, preserving nonresponse distinctions and valid Australian categories.

In [ ]:
clean_df = pd.read_csv(EXPORT, low_memory=False)
assert clean_df.shape == (28_333, 70), f"Unexpected P1 shape {clean_df.shape}"
assert clean_df.id.is_unique and clean_df.id.notna().all()
assert clean_df.country_name.nunique() == 24
assert "educ3" not in clean_df, "Unexpected educ3: require separately documented P2 integration"
assert int(clean_df.age_band.isna().sum()) == 185
assert int(clean_df.age_band_recovered.fillna(False).sum()) == 111
assert clean_df.age_band.value_counts().to_dict() == {"18–29":6068,"30–49":9787,"50–64":6596,"65+":5697}
assert int(clean_df.loc[clean_df.country_name.eq("Australia"), "age_band"].isna().sum()) == 2
assert int(clean_df.loc[clean_df.country_name.ne("Australia"), "age_band"].isna().sum()) == 183
assert int(clean_df.gender_clean.eq(1).sum()) == 15107
assert int(clean_df.gender_clean.eq(2).sum()) == 13203
assert int(clean_df.gender_clean.eq(3).sum()) == 21
assert int(clean_df.gender_clean.isna().sum()) == 2
assert int(clean_df.internetfreq.eq(6).sum()) == 3303
assert int(clean_df.age.eq(97).sum()) == 3
assert int(clean_df.ai_heard.notna().sum()) == 27525
assert int(clean_df.ai_cncexc.notna().sum()) == 26369
assert int(clean_df.aireg_us.eq(8).sum()) == 3112
assert int(clean_df.aireg_china.eq(8).sum()) == 3376
assert int(clean_df.aireg_eu.eq(8).sum()) == 4019
assert int(clean_df.aireg_country.eq(8).sum()) == 2946
assert clean_df.weight.gt(0).all() and clean_df.weight.notna().all()
assert len([c for c in clean_df.columns if c.startswith("d_educ_")]) == 27
print("PASS: 28,333 unique respondents, 70 P1 columns, 24 countries")
print("PASS: 28,148 known age bands; 185 missing (183 outside Australia + 2 in Australia)")
print("PASS: 111 recovered age bands; 21 Australian Other genders; 3 valid 97+ ages")
print("PASS: original education fields kept, educ3 still pending")

## C. Compare provisional data with the original CSV

The original CSV is retained separately. Compare by respondent ID (not row position), validate weights, and show that labelled refusals were recoded only in the clean export.

In [ ]:
raw_df = pd.read_csv(SOURCE_CSV, usecols=["id", "country", "weight", "ai_heard", "ai_cncexc",
                                           "aireg_us", "aireg_china", "aireg_eu", "aireg_country",
                                           "internetfreq", "age"], low_memory=False,
                     na_values=[" "], skipinitialspace=True)
assert len(raw_df) == 28_333 and raw_df.id.is_unique
joined = clean_df.merge(raw_df, on="id", how="outer", validate="one_to_one", indicator=True, suffixes=("_clean","_raw"))
assert joined._merge.eq("both").all() and len(joined) == 28_333
assert joined.country_clean.eq(joined.country_raw).all()
assert np.allclose(joined.weight_clean, joined.weight_raw, rtol=0, atol=1e-10)
for column, missing_codes in {"ai_heard":{8,9}, "ai_cncexc":{8,9},
                              "aireg_us":{9},"aireg_china":{9},"aireg_eu":{9},"aireg_country":{9},
                              "internetfreq":{8,9},"age":{98,99}}.items():
    expected = joined[f"{column}_raw"].mask(joined[f"{column}_raw"].isin(missing_codes))
    observed = joined[f"{column}_clean"]
    assert np.allclose(expected, observed, equal_nan=True, rtol=0, atol=0), f"Unexpected recoding: {column}"
print("PASS: 28,333 original/cleaned IDs align; weights and labels agree")
print("PASS: all 8 analytical variables recoded ONLY at their verified missing codes")
print("PASS: valid Q32 Not sure (8), internet nonuse (6), and age 97+ preserved")

## D. Independently reproduce 192 published Pew topline cells

Use **original responses** and survey weights so combined DK/refusal responses are counted in the denominators. The reference table records published whole percentages.

In [ ]:
ref = pd.read_csv(REFERENCE)
assert ref.shape[0] == 192 and set(ref.columns) >= {"country", "question", "code", "expected"}
ref_actual = joined[["id", "country_name", "weight_raw", "ai_heard_raw", "ai_cncexc_raw"]]
checks = []
for item in ref.itertuples(index=False):
    group = ref_actual.loc[ref_actual.country_name.eq(item.country)]
    assert not group.empty, f"No respondents: {item.country}"
    values = group[f"{item.question}_raw"]
    selected = values.isin([8, 9]) if int(item.code) == 8 else values.eq(int(item.code))
    exact = 100.0 * group.loc[selected,"weight_raw"].sum() / group.weight_raw.sum()
    rounded = int(np.floor(exact + 0.5))
    checks.append((rounded, int(item.expected), item.country, item.question, int(item.code)))
failures = [row for row in checks if row[0] != row[1]]
assert not failures, f"Official topline mismatches: {failures[:10]}"
assert len(checks) == 192
uk = ref_actual.loc[ref_actual.country_name.eq("UK")]
uk_w = 100 * uk.loc[uk.ai_heard_raw.eq(1), "weight_raw"].sum()/uk.weight_raw.sum()
uk_n = 100 * uk.ai_heard_raw.eq(1).mean()
assert round(uk_w) == 41 and round(uk_n) == 44
print(f"PASS: {len(checks)}/192 published Q30/Q31 cells reproduced")
print(f"UK Q30 A lot: {uk_w:.4f}% weighted -> 41%; {uk_n:.4f}% unweighted -> 44%")

## E. Survey-weight concentration and small age-group cells

Kish effective sample sizes are **weight-only approximations**, not full complex-design standard errors.

In [ ]:
by_country = []
for country, group in clean_df.groupby("country_name"):
    w = group.weight.to_numpy(dtype=float)
    deff_w = len(w) * np.sum(w * w) / np.sum(w)**2
    by_country.append((country, len(w), deff_w, len(w)/deff_w))
deffects = pd.DataFrame(by_country, columns=["country","n","deff_weight_only","kish_effective_n"])
for country, expected in {"Italy":2.1754,"India":2.0676,"Australia":1.2128}.items():
    actual = float(deffects.set_index("country").loc[country,"deff_weight_only"])
    assert abs(actual-expected)<0.0001, f"Weight-only DEFF differs: {country}"
cell_sizes = clean_df.dropna(subset=["age_band"]).groupby(["country_name","age_band"]).size()
assert len(cell_sizes)==96 and int(cell_sizes.lt(100).sum())==9
assert int(cell_sizes.loc[("India","18–29")]) == 1085
assert int(cell_sizes.loc[("Hungary","18–29")]) == 80
assert int(cell_sizes.loc[("South Korea","18–29")]) == 80
assert int(cell_sizes.loc[("Nigeria","65+")]) == 20
print("PASS: Kish estimates verified; 96 country-age cells, nine below 100 respondents")
print(deffects.loc[deffects.country.isin(["Italy","India","Australia"])].round(3).to_string(index=False))

## F. Reproduce weighted age-gap sensitivity checks

Age gap is Q30 “A lot” among ages 18–29 minus ages 65+, calculated separately for each country. The baseline includes nonresponse in the denominator. Differences are descriptive, not causal or inferential.

In [ ]:
from src.cleaning import awareness_gap, sensitivity_checks
results = sensitivity_checks(clean_df)
summary = results.groupby("scenario").change_pp.agg(mean_absolute_pp=lambda x: x.abs().mean())
expected_effects = {"exclude_dk":0.505791,"unweighted":2.957394,
                    "without_recovered_age":0.205653,
                    # cleaning.py DROPS the 21 Australian Other people here (Task 27),
                    # so this equals the gender-recorded result (Australia -1.06 / 24), not 0.
                    "exclude_australia_other_gender":0.044270}
for name, expected in expected_effects.items():
    actual = float(summary.loc[name,"mean_absolute_pp"])
    assert abs(actual-expected)<0.00001, f"Sensitivity differs: {name}={actual}"
print("PASS: all four Q30 country-age sensitivity scenarios reproduced")
print(summary.round(6).to_string())

## G. Final checks and P2 handoff

The P1 notebook is reproducible *without P2*. Education harmonisation remains an explicitly pending separate integration task. Running this notebook does not overwrite private datasets.

In [ ]:
assert len(clean_df) == 28_333 and len(joined) == 28_333
assert clean_df.columns.is_unique and clean_df.id.is_unique
assert not (CLEAN / "education_from_p2.csv").is_file() or "educ3" not in clean_df,     "P1 export should remain untouched by P2"
print("TASK 31 NOTEBOOK PASSED: fresh-kernel P1 analysis runs top to bottom")
print("P2 educ3 integration remains pending; Task 30 not falsely marked complete")